In [2]:
from dotenv import load_dotenv
import os

load_dotenv()
google_json = os.getenv("GOOGLE_APPLICATION_CREDENTIALS")
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = google_json

In [3]:
from google.cloud import storage

def upload_to_gcs(local_path, bucket_name, gcs_path):
    client = storage.Client()
    bucket = client.bucket(bucket_name)
    blob = bucket.blob(gcs_path)
    blob.upload_from_filename(local_path)
    return f"gs://{bucket_name}/{gcs_path}"

# CHANGE THESE
LOCAL_PDF = r"E:\Github\LawAssistant\triplet_extraction\data\luat_dat_dai\files\VanBanGoc_59.signed.pdf"
BUCKET_NAME = "vak_ocr_pdf"
GCS_INPUT_PATH = "pdfs/59_2020_QH14.pdf"

upload_to_gcs(LOCAL_PDF, BUCKET_NAME, GCS_INPUT_PATH)

Uploaded to gs://vak_ocr_pdf/pdfs/59_2020_QH14.pdf


In [6]:
def async_detect_document(gcs_source_uri, gcs_destination_uri):
    import json
    import re
    from google.cloud import vision
    from google.cloud import storage

    storage_client = storage.Client()

    # Parse destination bucket/prefix
    match = re.match(r"gs://([^/]+)/(.+)", gcs_destination_uri)
    bucket_name = match.group(1)
    prefix = match.group(2)

    bucket = storage_client.bucket(bucket_name)

    # 🔍 CHECK IF OCR OUTPUT ALREADY EXISTS
    existing_blobs = [
        blob for blob in bucket.list_blobs(prefix=prefix)
        if blob.name.endswith(".json")
    ]

    if existing_blobs:
        print("OCR output already exists → loading from GCS")
    else:
        print("No OCR output found → running OCR")

        mime_type = "application/pdf"
        batch_size = 2

        vision_client = vision.ImageAnnotatorClient()

        feature = vision.Feature(
            type_=vision.Feature.Type.DOCUMENT_TEXT_DETECTION
        )

        input_config = vision.InputConfig(
            gcs_source=vision.GcsSource(uri=gcs_source_uri),
            mime_type=mime_type
        )

        output_config = vision.OutputConfig(
            gcs_destination=vision.GcsDestination(uri=gcs_destination_uri),
            batch_size=batch_size
        )

        request = vision.AsyncAnnotateFileRequest(
            features=[feature],
            input_config=input_config,
            output_config=output_config
        )

        operation = vision_client.async_batch_annotate_files(
            requests=[request]
        )

        operation.result(timeout=900)

        # Reload blobs after OCR
        existing_blobs = [
            blob for blob in bucket.list_blobs(prefix=prefix)
            if blob.name.endswith(".json")
        ]

    # 🔑 SORT OUTPUT FILES BY PAGE NUMBER
    def start_page(blob_name):
        m = re.search(r"output-(\d+)-to-\d+\.json", blob_name)
        return int(m.group(1)) if m else 0

    existing_blobs.sort(key=lambda b: start_page(b.name))

    # 📄 READ ALL PAGES
    full_text = []
    page_count = 0

    for blob in existing_blobs:
        response = json.loads(blob.download_as_text())
        for page in response["responses"]:
            page_count += 1
            text = page.get("fullTextAnnotation", {}).get("text", "")
            if text.strip():
                full_text.append(text)

    print(f"Total pages loaded: {page_count}")

    return "\n".join(full_text)

In [7]:
gcs_source_uri = "gs://vak_ocr_pdf/pdfs/59_2020_QH14.pdf"
gcs_destination_uri = "gs://vak_ocr_pdf/ocr_outputs/"

text = async_detect_document(
    gcs_source_uri,
    gcs_destination_uri
)

print(text[:2000])  # preview first 2000 chars
with open("../../src/triplet_extraction/59_2020_QH14_ocr.txt", "w", encoding="utf-8") as f:
    f.write(text)

print("Saved 59_2020_QH14_ocr.txt")

OCR output already exists → loading from GCS
Total pages loaded: 70
VGP
Ký bởi: Cổng Thông tin điện tử Chính phủ
Email: thongtinchinhphu@chinhphu.vn
Cơ quan: Văn phòng Chính phủ
CÔNG THÔNG TIN ĐIỆN TỬ CHÍNH PHỦ •Thời gian ký: 08.07.2020 15:53:10 +07:00
QUỐC HỘI
Luật số: 59/2020/QH14
CỘNG HÒA XÃ HỘI CHỦ NGHĨA VIỆT NAM
Độc lập - Tự do - Hạnh phúc
LUẬT
DOANH NGHIỆP
Căn cứ Hiến pháp nước Cộng hòa xã hội chủ nghĩa Việt Nam;
Quốc hội ban hành Luật Doanh nghiệp.
Chương I
NHỮNG QUY ĐỊNH CHUNG
Điều 1. Phạm vi điều chỉnh
Luật này quy định về việc thành lập, tổ chức quản lý, tổ chức lại, giải thể
và hoạt động có liên quan của doanh nghiệp, bao gồm công ty trách nhiệm hữu
hạn, công ty cổ phần, công ty hợp danh và doanh nghiệp tư nhân; quy định về
nhóm công ty.
Điều 2. Đối tượng áp dụng
1. Doanh nghiệp.
2. Cơ quan, tổ chức, cá nhân có liên quan đến việc thành lập, tổ chức quản
lý, tổ chức lại, giải thể và hoạt động có liên quan của doanh nghiệp.
Điều 3. Áp dụng Luật Doanh nghiệp và luật khác
Trường